# 04 — MongoDB Atlas: NoSQL Database for NorthStar

**Module:** Databases and Analytics
**Case study:** NorthStar Urban Mobility & Logistics
**Learning outcomes covered:** LO2 (build NoSQL DB), LO3 (indexing & query optimisation), LO4 (analytics).

This notebook implements the **NoSQL design** justified in the report. It connects to MongoDB Atlas
using PyMongo, creates 4 collections, ingests data (including a nested `customer_cases` collection
that integrates journey + complaint + event-history context), demonstrates CRUD, runs aggregation
pipelines for analytics, and benchmarks indexes.

> **Setup before running:**
> 1. Create a free MongoDB Atlas cluster: https://www.mongodb.com/cloud/atlas
> 2. Add your IP to the network access list (or `0.0.0.0/0` for testing).
> 3. Create a database user; copy the connection string (`mongodb+srv://...`).
> 4. In Colab, store it via `from google.colab import userdata; userdata.get('MONGO_URI')`,
>    or paste it into the cell below (do **not** commit secrets to GitHub).


## 1. Install & connect

In [ ]:
# !pip install pymongo[srv] dnspython pandas --quiet

import os, json, time
import pandas as pd
from pymongo import MongoClient, ASCENDING, DESCENDING, TEXT
from pymongo.errors import OperationFailure

# Replace with your Atlas URI. For Colab, prefer userdata.get('MONGO_URI').
MONGO_URI = os.environ.get("MONGO_URI", "mongodb://localhost:27017")

client = MongoClient(MONGO_URI)
db = client["northstar"]
print("Connected. Existing collections:", db.list_collection_names())


## 2. Schema design — what stays relational, what becomes a document?

The design separates **stable reference data** (relational-style, 1 document per entity) from
**evolving operational records** (rich nested documents).

| Collection         | Shape       | Purpose                                                      |
|--------------------|-------------|--------------------------------------------------------------|
| `vehicles`         | flat        | Fleet reference data; rarely changes                         |
| `drivers`          | flat        | Driver reference; small profile updates only                 |
| `journeys`         | flat        | One document per journey; high write volume; analytical ops  |
| `customer_cases`   | **nested**  | One document per customer case, embedding journey context, event history, tags  |

**Why `customer_cases` is nested:**
Today the company stores journey, complaint, exception, and event-history data in 4 different
systems. The case study explicitly says staff struggle to combine them. A document model lets a
single read return the full case — customer, linked journey, severity, all events — without joins.
This directly supports the integrated operational view the board asked for.

**Why journeys stay flat (not embedded under customers):**
Journeys are written far more often than customers are read; embedding journeys would make customer
documents grow unbounded and slow common queries. Journey IDs are referenced from cases when
context is needed.


## 3. Drop & recreate collections

In [ ]:
for c in ["vehicles","drivers","journeys","customer_cases"]:
    db.drop_collection(c)
print("Dropped (if existed). Now creating fresh.")


## 4. Ingest reference & operational data

In [ ]:
DATA = "data"

vehicles = pd.read_csv(f"{DATA}/vehicles.csv").to_dict("records")
drivers  = pd.read_csv(f"{DATA}/drivers.csv").to_dict("records")
journeys = pd.read_csv(f"{DATA}/journeys.csv").to_dict("records")

print(f"Inserting {len(vehicles):,} vehicles ...")
db.vehicles.insert_many(vehicles)
print(f"Inserting {len(drivers):,} drivers ...")
db.drivers.insert_many(drivers)
print(f"Inserting {len(journeys):,} journeys ...")
db.journeys.insert_many(journeys)
print("Done.")


## 5. Ingest the nested `customer_cases` collection

This is the centrepiece of the NoSQL design — each document embeds the full case context.

In [ ]:
with open(f"{DATA}/customer_cases.json") as f:
    cases = json.load(f)

print(f"Inserting {len(cases):,} nested customer cases ...")
db.customer_cases.insert_many(cases)
print("Sample document structure:")
print(json.dumps(db.customer_cases.find_one({}, {"_id":0}), indent=2)[:1200])


## 6. CRUD demonstration

In [ ]:
# CREATE — log a new case
new_case = {
    "case_id": "CASE-DEMO-001",
    "customer": {"customer_id": "C00001", "city": "London", "loyalty_tier": "Gold"},
    "linked_journey": {"journey_id": "J999999", "service_type": "LastMileDelivery",
                       "zone_id": "Z001", "status": "Failed", "delay_minutes": 25},
    "category": "FailedDelivery",
    "severity": "High",
    "submitted_date": "2026-05-01",
    "resolved": False,
    "events": [
        {"seq": 1, "event_type": "MessageReceived", "actor": "Customer",
         "note": "Driver never arrived"}
    ],
    "tags": ["urgent","gps-mismatch"]
}
db.customer_cases.insert_one(new_case)
print("Created CASE-DEMO-001")


In [ ]:
# READ — fetch and project
db.customer_cases.find_one({"case_id":"CASE-DEMO-001"},
                           {"case_id":1,"severity":1,"events":1,"_id":0})


In [ ]:
# UPDATE — push a new event into the embedded array, mark resolved
db.customer_cases.update_one(
    {"case_id":"CASE-DEMO-001"},
    {"$push": {"events": {"seq": 2, "event_type":"AutoResolved",
                          "actor":"System", "note":"Refund issued"}},
     "$set":  {"resolved": True}}
)
db.customer_cases.find_one({"case_id":"CASE-DEMO-001"},
                           {"events":1,"resolved":1,"_id":0})


In [ ]:
# DELETE — remove the demo case (keep collection clean)
db.customer_cases.delete_one({"case_id":"CASE-DEMO-001"})
print("Deleted demo case.")


## 7. Analytical aggregation pipelines

These answer the board's questions directly from the document store — no joins to relational tables required.

In [ ]:
# 7.1 — Worst-performing zones from cases (failure / late context, embedded)
pipe = [
    {"$group": {
        "_id": "$linked_journey.zone_id",
        "cases": {"$sum": 1},
        "avg_delay": {"$avg": "$linked_journey.delay_minutes"},
        "high_severity_pct": {"$avg": {"$cond": [{"$eq": ["$severity","High"]}, 1, 0]}}
    }},
    {"$sort": {"cases": -1}},
    {"$limit": 10}
]
list(db.customer_cases.aggregate(pipe))


In [ ]:
# 7.2 — Complaint-category mix per city
pipe = [
    {"$group": {
        "_id": {"city": "$customer.city", "category": "$category"},
        "count": {"$sum": 1}
    }},
    {"$sort": {"_id.city": 1, "count": -1}}
]
res = list(db.customer_cases.aggregate(pipe))
pd.DataFrame([{"city": r["_id"]["city"], "category": r["_id"]["category"], "count": r["count"]}
              for r in res]).head(20)


In [ ]:
# 7.3 — Cases tagged 'gps-mismatch' AND linked to journeys with manual_override = 1
# (cross-collection lookup demonstrating $lookup)
pipe = [
    {"$match": {"tags": "gps-mismatch"}},
    {"$lookup": {
        "from": "journeys",
        "localField": "linked_journey.journey_id",
        "foreignField": "journey_id",
        "as": "journey_doc"
    }},
    {"$unwind": "$journey_doc"},
    {"$match": {"journey_doc.manual_override": 1}},
    {"$project": {"_id":0, "case_id":1, "severity":1,
                  "delay": "$journey_doc.delay_minutes",
                  "service": "$journey_doc.service_type"}},
    {"$limit": 5}
]
list(db.customer_cases.aggregate(pipe))


In [ ]:
# 7.4 — Average events per case by severity (depth of investigation indicator)
pipe = [
    {"$project": {"severity": 1, "n_events": {"$size": "$events"}}},
    {"$group": {"_id": "$severity",
                "avg_events": {"$avg": "$n_events"},
                "cases": {"$sum": 1}}},
    {"$sort": {"_id": 1}}
]
list(db.customer_cases.aggregate(pipe))


## 8. Indexing & query optimisation

The case study explicitly mentions that current systems struggle to query case histories quickly. We index the fields most used in filters/joins/aggregations.

In [ ]:
# Single-field, compound, multikey, and text indexes
db.customer_cases.create_index([("customer.customer_id", ASCENDING)], name="ix_customer_id")
db.customer_cases.create_index([("linked_journey.zone_id", ASCENDING),
                                ("severity", ASCENDING)], name="ix_zone_severity")
db.customer_cases.create_index([("submitted_date", DESCENDING)], name="ix_submitted_date")
db.customer_cases.create_index([("tags", ASCENDING)], name="ix_tags_multikey")  # array → multikey

db.journeys.create_index([("zone_id", ASCENDING)], name="ix_j_zone")
db.journeys.create_index([("status", ASCENDING)], name="ix_j_status")
db.journeys.create_index([("driver_id", ASCENDING),
                          ("date", DESCENDING)], name="ix_j_driver_date")

# Text index on event notes for full-text search
try:
    db.customer_cases.create_index([("events.note", TEXT)], name="ix_event_note_text")
except OperationFailure as e:
    print("Text index already exists or failed:", e)

print("Indexes on customer_cases:")
for ix in db.customer_cases.list_indexes(): print(" ", ix["name"], ix["key"])
print("Indexes on journeys:")
for ix in db.journeys.list_indexes(): print(" ", ix["name"], ix["key"])


### 8.1 Measure: explain plan + timing before/after

MongoDB's `explain()` shows the query plan. We compare a query that scans the whole collection (`COLLSCAN`) vs one that uses an index (`IXSCAN`).

In [ ]:
# Simulate a non-indexed lookup using a hint that forces a full scan
q = {"linked_journey.zone_id": "Z015", "severity": "High"}

print(">>> WITH index (default planner)")
plan_with = db.customer_cases.find(q).explain()["queryPlanner"]["winningPlan"]
print(json.dumps(plan_with, indent=2)[:500])

# Time it (run a few times)
def time_query(coll, q, hint=None, runs=10):
    times = []
    for _ in range(runs):
        t = time.perf_counter()
        cur = coll.find(q)
        if hint: cur = cur.hint(hint)
        list(cur)
        times.append(time.perf_counter() - t)
    return sum(times)/len(times)*1000  # ms

ms_with = time_query(db.customer_cases, q)
ms_without = time_query(db.customer_cases, q, hint=[("$natural", 1)])

print(f"\nAvg query time WITH index    : {ms_with:.2f} ms")
print(f"Avg query time WITHOUT index : {ms_without:.2f} ms")
print(f"Speed-up factor              : {ms_without/ms_with:.1f}x")


### 8.2 Text search example

In [ ]:
list(db.customer_cases.find(
    {"$text": {"$search": "refund neighbour"}},
    {"case_id":1,"events.note":1,"_id":0}
).limit(5))


## 9. Justification of design decisions

* **Embed events inside cases.** Events are read with their case ~always; they don't grow without bound (case study mentions sequences up to a few dozen events). Embedding avoids costly joins.
* **Reference customer & journey rather than embed them.** Both are reused across many cases; embedding would duplicate data and create update anomalies.
* **Array indexes (`multikey`) on `tags`.** Tag-based lookups (e.g., `gps-mismatch`, `vip`) are common operational queries and cheap on multikey indexes.
* **Compound index `zone_id + severity`.** Matches the most common analytics access pattern: 'show me high-severity cases by zone'.
* **Text index on `events.note`.** Lets analysts surface free-text patterns (e.g. driver excuses, refund mentions) without redesign.
* **Journeys stay flat.** They are append-heavy, queried analytically, and benefit from a wide compound index strategy more than embedding.

## 10. Summary

The NoSQL design closes a real gap in NorthStar's environment. Today, answering 'what happened in this case?' requires four systems and manual reconciliation. With this design, a single `find_one` returns the full picture, and aggregations across thousands of cases can be expressed in a few lines.
